In [0]:
from pyspark.sql import functions as F

BRONZE_TABLE = "dbr_dev.brazilian_ecommerce_bronze.brz_orders"
ORDERS_PATH = "/Volumes/dbr_dev/brazilian_ecommerce_bronze/landing/orders/orders.csv"
ORDER_ITEMS_PATH = "/Volumes/dbr_dev/brazilian_ecommerce_bronze/landing/order_items/order_items.csv"

In [0]:
orders_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(ORDERS_PATH)
)

order_items_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(ORDER_ITEMS_PATH)
)


In [0]:
joined_orders_df = orders_df.join(order_items_df, on="order_id", how="inner")


In [0]:
historical_orders_df = (
    joined_orders_df
    .groupBy("order_id", "customer_id", "product_id", "order_purchase_timestamp")
    .agg(
        F.count("*").cast("int").alias("quantity"),
        F.round(F.sum("price"), 2).alias("price")
    )
)

In [0]:
historical_orders_df = (
    historical_orders_df
    .withColumnRenamed("order_purchase_timestamp", "order_timestamp")
    .withColumn("discount_code", F.lit(None).cast("string"))
    .withColumn("kafka_partition", F.lit(None).cast("int"))
    .withColumn("kafka_offset", F.lit(None).cast("long"))
    .withColumn("source", F.lit("batch_orders"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)


In [0]:
if spark.catalog.tableExists(BRONZE_TABLE):
    display(spark.sql(f"SELECT * FROM {BRONZE_TABLE} LIMIT 5"))
    spark.sql(f"""
        SELECT source, COUNT(*) AS total
        FROM {BRONZE_TABLE}
        GROUP BY source
    """).show()


In [0]:
if spark.catalog.tableExists(BRONZE_TABLE):
    spark.sql(f"""DELETE FROM {BRONZE_TABLE} WHERE source = 'batch_orders'""")

In [0]:
(
    historical_orders_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(BRONZE_TABLE)
)

In [0]:
spark.sql(f"""
    SELECT source, COUNT(*) AS total
    FROM {BRONZE_TABLE}
    GROUP BY source
""").show()